# SVG Inference — Load Merged Model & Generate Submission

Inference-only notebook. Loads the pre-trained merged model from Kaggle kernel output and generates `submission.csv` for the 1,000 test prompts.

**Before running:** Add the training kernel output as a data source in Kaggle (Add Data → Kernel Outputs → search `ryanmfleishman/dl-llama-version`). The merged model will then be at `/kaggle/input/dl-llama-version/qwen25_coder_svg_lora_merged`.

In [1]:
%pip install -q unsloth transformers accelerate peft bitsandbytes pandas lxml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.2/403.2 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Paths
MERGED_MODEL_DIR = "/kaggle/input/datasets/ryanmfleishman/dl-training/qwen25_coder_svg_lora_merged"
TEST_PATH        = "/kaggle/input/competitions/dl-spring-2026-svg-generation/test.csv"
SUBMISSION_PATH  = "/kaggle/working/submission.csv"
MAX_NEW_TOKENS   = 1024  # was 256 — SVGs need room to breathe

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Model path exists: {os.path.isdir(MERGED_MODEL_DIR)}")

Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Model path exists: True


In [ ]:
from lxml import etree
import re, xml.etree.ElementTree as ET

# SVG constraint constants
MAX_SVG_CHARS     = 16_000
MAX_PATH_ELEMENTS = 256
ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse", "line",
    "polyline", "polygon", "defs", "use", "symbol", "clipPath",
    "mask", "linearGradient", "radialGradient", "stop", "text",
    "tspan", "title", "desc", "style", "pattern", "marker", "filter",
}

_URL_REF_RE = re.compile(r"url\(#([^)]+)\)")


def _local_tag(element):
    tag = element.tag
    return tag.split("}", 1)[-1] if isinstance(tag, str) and "}" in tag else tag


def is_valid_svg(svg_text):
    if not svg_text or not isinstance(svg_text, str):
        return False
    if len(svg_text) > MAX_SVG_CHARS:
        return False
    svg_text = svg_text.strip()
    if not svg_text.lower().startswith("<svg"):
        return False
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return False
    if _local_tag(root) != "svg":
        return False
    path_count = 0
    for el in root.iter():
        tag = _local_tag(el)
        if tag not in ALLOWED_TAGS:
            return False
        if tag == "path":
            path_count += 1
    if path_count > MAX_PATH_ELEMENTS:
        return False
    return True


def normalize_dimensions(svg_text):
    """
    Force width/height to 256. Keep original viewBox so the internal
    coordinate system scales correctly to the 256x256 canvas.
    Do NOT add preserveAspectRatio — the scorer's renderer interprets
    the SVG spec default differently from the explicit attribute.
    """
    if not svg_text:
        return svg_text
    try:
        parser = etree.XMLParser(recover=True)
        root = etree.fromstring(svg_text.encode("utf-8"), parser)
    except Exception:
        return svg_text
    if _local_tag(root) != "svg":
        return svg_text
    root.set("width", "256")
    root.set("height", "256")
    if not root.get("viewBox"):
        root.set("viewBox", "0 0 256 256")
    return etree.tostring(root, encoding="unicode")


# ---------------------------------------------------------------------------
# SVG repair helpers
# ---------------------------------------------------------------------------

def _collect_defined_ids(root):
    ids = set()
    for el in root.iter():
        id_val = el.get("id")
        if id_val:
            ids.add(id_val)
    return ids


def _fix_broken_url_refs(root, defined_ids):
    """Replace url(#id) that point to undefined IDs with safe fallbacks."""
    url_attrs_color  = {"fill", "stroke"}
    url_attrs_remove = {"filter", "clip-path", "mask"}
    for el in root.iter():
        for attr in list(el.attrib):
            val = el.get(attr, "")
            m = _URL_REF_RE.fullmatch(val.strip())
            if not m:
                continue
            ref_id = m.group(1)
            if ref_id in defined_ids:
                continue
            local = attr.split("}", 1)[-1] if "}" in attr else attr
            if local in url_attrs_color:
                el.set(attr, "black")
            elif local in url_attrs_remove:
                del el.attrib[attr]


def _prune_invisible_elements(root):
    """Remove definitively invisible elements to avoid rendering noise."""
    to_remove = []
    for el in root.iter():
        if _local_tag(el) == "svg":
            continue
        if el.get("opacity") == "0" or el.get("display") == "none":
            to_remove.append(el)
    for el in to_remove:
        parent = el.getparent()
        if parent is not None:
            parent.remove(el)


def repair_svg(raw_text):
    """
    Attempt to salvage malformed/truncated model output.
    Steps:
      1. Locate opening <svg tag
      2. Append </svg> if truncated (strip trailing incomplete tag first)
      3. lxml recovery parse
      4. Fix required root attributes
      5. Remove disallowed tags
      6. Trim excess <path> elements
      7. Fix broken url(#...) references
      8. Prune invisible elements
    """
    if not raw_text or not isinstance(raw_text, str):
        return None

    text = raw_text.strip()

    # 1. Locate the opening <svg tag
    m = re.search(r"<svg[\s>]", text, re.IGNORECASE)
    if not m:
        return None
    text = text[m.start():]

    # 2. If closing tag is missing the model was truncated — append it
    if not re.search(r"</svg\s*>", text, re.IGNORECASE):
        # Strip any trailing incomplete open tag before closing
        text = re.sub(r"<[^>]*$", "", text)
        text = text.rstrip() + "</svg>"

    # 3. lxml recovery parse
    try:
        parser = etree.XMLParser(recover=True, remove_comments=True)
        root = etree.fromstring(text.encode("utf-8"), parser)
    except Exception:
        return None
    if root is None or _local_tag(root) != "svg":
        return None

    # 4. Ensure required root attributes
    if "xmlns" not in root.attrib:
        root.set("xmlns", "http://www.w3.org/2000/svg")
    for attr, val in [("width", "256"), ("height", "256"), ("viewBox", "0 0 256 256")]:
        if attr not in root.attrib:
            root.set(attr, val)

    # 5. Remove disallowed tags
    for el in reversed(root.xpath(".//*")):
        if _local_tag(el) not in ALLOWED_TAGS:
            parent = el.getparent()
            if parent is not None:
                parent.remove(el)

    # 6. Trim excess <path> elements
    paths = root.xpath(".//*[local-name()='path']")
    if len(paths) > MAX_PATH_ELEMENTS:
        for el in paths[MAX_PATH_ELEMENTS:]:
            p = el.getparent()
            if p is not None:
                p.remove(el)

    # 7. Fix broken url(#...) refs
    _fix_broken_url_refs(root, _collect_defined_ids(root))

    # 8. Prune invisible elements
    _prune_invisible_elements(root)

    repaired = etree.tostring(root, encoding="unicode")
    return repaired if is_valid_svg(repaired) else None


# ---------------------------------------------------------------------------
# Prompt-aware fallback: only reached if generation AND repair both fail.
# ---------------------------------------------------------------------------
_COLOR_MAP = {
    "red": "red", "blue": "blue", "green": "green", "yellow": "yellow",
    "orange": "orange", "purple": "purple", "pink": "pink",
    "black": "black", "white": "white", "gray": "gray", "grey": "gray",
    "brown": "#8B4513", "cyan": "cyan", "magenta": "magenta",
    "gold": "gold", "silver": "silver", "teal": "teal", "navy": "navy",
}

def prompt_aware_fallback(prompt=""):
    p = prompt.lower()

    fill = "steelblue"
    for kw, color in _COLOR_MAP.items():
        if kw in p:
            fill = color
            break

    bg = "black" if any(w in p for w in ("dark", "night", "space", "black background")) else "white"

    if any(w in p for w in ("circle", "ball", "sphere", "sun", "moon", "dot")):
        shape = f'<circle cx="128" cy="128" r="80" fill="{fill}"/>'
    elif any(w in p for w in ("triangle", "pyramid", "mountain")):
        shape = f'<polygon points="128,32 224,214 32,214" fill="{fill}"/>'
    elif "star" in p:
        shape = f'<polygon points="128,28 148,88 212,88 160,124 180,188 128,150 76,188 96,124 44,88 108,88" fill="{fill}"/>'
    elif "heart" in p:
        shape = f'<path d="M128,195 C55,148 18,98 58,58 C78,38 108,43 128,68 C148,43 178,38 198,58 C238,98 201,148 128,195 Z" fill="{fill}"/>'
    elif any(w in p for w in ("square", "box", "rect", "flag", "window")):
        shape = f'<rect x="48" y="48" width="160" height="160" fill="{fill}"/>'
    elif any(w in p for w in ("ellipse", "oval")):
        shape = f'<ellipse cx="128" cy="128" rx="110" ry="65" fill="{fill}"/>'
    elif any(w in p for w in ("line", "diagonal", "stripe")):
        shape = f'<line x1="24" y1="128" x2="232" y2="128" stroke="{fill}" stroke-width="10"/>'
    elif any(w in p for w in ("text", "letter", "word", "number")):
        shape = f'<text x="128" y="148" font-size="64" text-anchor="middle" fill="{fill}">A</text>'
    else:
        shape = f'<circle cx="128" cy="128" r="80" fill="{fill}"/>'

    return (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        f'<rect width="256" height="256" fill="{bg}"/>'
        f'{shape}'
        f'</svg>'
    )


# Sanity checks
assert is_valid_svg(prompt_aware_fallback("a red circle")), "Fallback must be valid"
assert is_valid_svg(prompt_aware_fallback()), "Default fallback must be valid"
_fixed = normalize_dimensions('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 400 400" height="128" width="128"><circle cx="200" cy="200" r="180" fill="blue"/></svg>')
assert 'width="256"' in _fixed and 'height="256"' in _fixed, "normalize_dimensions must set 256x256"
assert 'preserveAspectRatio' not in _fixed, "normalize_dimensions must NOT set preserveAspectRatio"
print("SVG helpers ready.")

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_DIR,
    torch_dtype=torch.float16,
    device_map="cuda:0",
)
model.eval()

# torch.compile gives ~20-40% throughput improvement on T4 with a one-time
# ~30s compilation cost on the first generate call.
model = torch.compile(model, mode="reduce-overhead")

print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")

_tok = tokenizer("warmup", return_tensors="pt").to("cuda:0")
with torch.no_grad():
    _t0 = time.time()
    model.generate(**_tok, max_new_tokens=32, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    _elapsed = time.time() - _t0
print(f"Warmup (includes first-run compile): {_elapsed:.2f}s for 32 tokens  ({32/_elapsed:.1f} tok/s)")

# Second run shows true compiled speed
with torch.no_grad():
    _t0 = time.time()
    model.generate(**_tok, max_new_tokens=32, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    _elapsed = time.time() - _t0
print(f"Warmup (compiled, 2nd run):           {_elapsed:.2f}s for 32 tokens  ({32/_elapsed:.1f} tok/s)")

The tokenizer you are loading from '/kaggle/input/datasets/ryanmfleishman/dl-training/qwen25_coder_svg_lora_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

GPU memory used: 3.09 GB / 15.64 GB
Warmup (includes first-run compile): 2.28s for 32 tokens  (14.0 tok/s)
Warmup (compiled, 2nd run):           1.29s for 32 tokens  (24.9 tok/s)


In [ ]:
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from a natural language description. "
    "Output ONLY the SVG code — no explanation, no markdown, no code fences. "
    "Rules:\n"
    "- Root element must be <svg> with width=\"256\" height=\"256\" viewBox=\"0 0 256 256\" "
    "xmlns=\"http://www.w3.org/2000/svg\".\n"
    "- Always include a background <rect width=\"256\" height=\"256\" fill=\"...\"/> as the first child.\n"
    "- All coordinates and dimensions must be in the range 0–256.\n"
    "- Use <circle>, <rect>, <polygon>, <ellipse> for simple shapes. "
    "Reserve <path> only for complex curves that cannot be expressed with simpler elements.\n"
    "- Use only allowed tags. Keep total output under 16000 characters."
)

SVG_REGEX = re.compile(r"<svg.*?</svg>", flags=re.IGNORECASE | re.DOTALL)


def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""


def generate_svg(prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.3,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    input_len = inputs["input_ids"].shape[1]
    decoded = tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=True)

    # Priority 1: clean extraction
    svg = extract_svg(decoded)
    if is_valid_svg(svg):
        return normalize_dimensions(svg)

    # Priority 2: attempt repair
    repaired = repair_svg(decoded)
    if repaired:
        return normalize_dimensions(repaired)

    # Priority 3: prompt-aware fallback (already 256x256 with preserveAspectRatio)
    return prompt_aware_fallback(prompt)


# Smoke test
test_svg = generate_svg("a simple blue circle on a white background")
print(f"Smoke test — valid: {is_valid_svg(test_svg)}, length: {len(test_svg)} chars")
print(test_svg[:300])

In [6]:
test_df = pd.read_csv(TEST_PATH)
print(f"Test rows: {len(test_df)}")

all_ids     = test_df["id"].tolist()
all_prompts = test_df["prompt"].tolist()

rows = []
invalid_count = 0
t0 = time.time()

for i, (id_, prompt) in enumerate(zip(all_ids, all_prompts)):
    t_row = time.time()
    svg = generate_svg(prompt)
    row_sec = time.time() - t_row

    if not is_valid_svg(svg):
        invalid_count += 1
    rows.append({"id": id_, "svg": svg})

    elapsed = (time.time() - t0) / 60
    rate = (i + 1) / elapsed if elapsed > 0 else 0
    eta = (len(all_prompts) - i - 1) / rate if rate > 0 else 0
    print(f"  [{i+1:4d}/{len(all_prompts)}] {row_sec:.1f}s/row | {elapsed:.1f} min elapsed | ETA {eta:.1f} min | fallbacks: {invalid_count}")

sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)

elapsed_min = (time.time() - t0) / 60
print(f"\nSaved: {SUBMISSION_PATH}")
print(f"Rows: {len(sub_df)}")
print(f"Fallbacks: {invalid_count} ({100*invalid_count/len(sub_df):.1f}%)")
print(f"Runtime: {elapsed_min:.1f} min")
sub_df.head()

Test rows: 1000
  [   1/1000] 39.6s/row | 0.7 min elapsed | ETA 659.2 min | fallbacks: 0
  [   2/1000] 9.0s/row | 0.8 min elapsed | ETA 403.8 min | fallbacks: 0
  [   3/1000] 11.4s/row | 1.0 min elapsed | ETA 332.1 min | fallbacks: 0
  [   4/1000] 39.5s/row | 1.7 min elapsed | ETA 412.6 min | fallbacks: 0
  [   5/1000] 5.4s/row | 1.7 min elapsed | ETA 347.8 min | fallbacks: 0
  [   6/1000] 4.8s/row | 1.8 min elapsed | ETA 302.8 min | fallbacks: 0
  [   7/1000] 5.7s/row | 1.9 min elapsed | ETA 272.8 min | fallbacks: 0
  [   8/1000] 39.4s/row | 2.6 min elapsed | ETA 320.0 min | fallbacks: 0
  [   9/1000] 16.3s/row | 2.9 min elapsed | ETA 314.0 min | fallbacks: 0
  [  10/1000] 7.7s/row | 3.0 min elapsed | ETA 295.1 min | fallbacks: 0
  [  11/1000] 5.1s/row | 3.1 min elapsed | ETA 275.6 min | fallbacks: 0
  [  12/1000] 39.4s/row | 3.7 min elapsed | ETA 306.5 min | fallbacks: 0
  [  13/1000] 39.1s/row | 4.4 min elapsed | ETA 332.1 min | fallbacks: 0
  [  14/1000] 39.4s/row | 5.0 min elapsed

,id,svg
0,fa1d8fa7-080f-4269-a9cf-a17562c9a0ca,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
1,6eede943219547c22ac56085027d33cc,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
2,ea045c7a247166f061ce504d9b7ccaab,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
3,8fe82f3af89e487b31236ca829c3f071,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
4,600464e4d92c75338462271a09b3f176,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
